# 10-fold training and the MRHydraET fallback — root cause analysis

Questions:

1. How does TSCGlue's 10-fold training work?
2. On which datasets does the fallback (`MRHydraET`) run instead of the stack, and why?
3. What does it cost, and can it be avoided?

Everything below is read from the installed `tscglue` (`.venv/.../tscglue/models.py`, `utils.py`,
`fallback.py`) and replayed offline: fold splitting only needs `y` and the seeds, so which runs fall
back can be reproduced exactly without fitting any model. The replay is then checked against the
real prediction files in `model_results/`.

## 1. How the 10-fold pipeline works

`LokyStackerV10Base._fit` (inherited by `TSCGlueEnhancedV4`), in order:

1. **Singleton guard.** If any class has fewer than 2 training cases → fallback immediately.
2. **Features.** Every feature transformer (multirocket, hydra, quant, rdst, rstsf, weasel, mantis,
   chronos2) is fit **once on the full training set, without `y`**, and the transformed arrays are
   cached. The folds below only ever re-train the classifier heads, never the transformers.
3. **Level 0.** Each base model spec gets its *own* fold seed (`spec.fold_seeds`, derived from
   `random_state`), so every base model sees a different 10-fold split.
   `generate_folds → utils.get_folds` builds the split:
   ```python
   try:
       StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)   # preferred
   except ValueError:
       KFold(n_splits=10, shuffle=True, random_state=seed)             # silent fallback
   ```
   Each fold trains a head on 9/10 and predicts the held-out 1/10 → out-of-fold (OOF) probabilities.
   After every fold, `_missing_classes` checks the head's `classes_`; **if a fold model does not
   know every class → fallback** (the OOF matrix has one column per class and cannot be built).
4. **Level 1 (stackers).** A new split (`stacker_fold_seed`) is drawn, and every stacker is 10-fold
   trained on the level-0 OOF probability matrix. Same missing-class check → fallback. A NaN in the
   OOF matrix also → fallback.
5. **Level 2 (`best` only).** The meta stacker gets one more split over level-0 + level-1 OOF.
6. **Predict.** Each base model = mean of its 10 fold models; the stackers see those means. If
   `fallback.pkl` exists, `_predict_proba` ignores the stack and serves the fallback instead.

The fallback (`TSCGlueEnhancedV3._fit_fallback`) is `MRHydraET(random_state=random_state)`:
multirocket + hydra features → one ExtraTrees, no folds. It depends only on `random_state`, so
**every preset and eval metric returns the same predictions when it falls back.**

In [1]:
import contextlib
import io
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from aeon.datasets import load_from_ts_file
from aeon.datasets.tsc_datasets import univariate_equal_length
from sklearn.model_selection import StratifiedKFold
from tscglue import utils as tg_utils
from tscglue.models import TSCGlueEnhancedV4, generate_folds

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..")
DATA = PROJECT_ROOT / "data"
MODEL_RESULTS = PROJECT_ROOT / "model_results"
DATASETS = sorted(univariate_equal_length)
RESAMPLES = range(16)  # resamples present in model_results for the TSCGlue runs
PRESETS = ["low", "medium", "high", "best"]
K = 10

In [2]:
# Fold models trained per fit, by preset
rows = []
for preset in PRESETS:
    clf = TSCGlueEnhancedV4(random_state=0, preset=preset)
    n_base, n_stack, n_meta = len(clf.model_specs), len(clf.stacking_models), len(clf.meta_stacking_models)
    rows.append({
        "preset": preset,
        "base models": n_base,
        "stackers": n_stack,
        "meta": n_meta,
        "distinct 10-fold splits": n_base + 1 + (n_meta > 0),
        "fold models trained": K * (n_base + n_stack + n_meta),
    })
preset_table = pd.DataFrame(rows).set_index("preset")
preset_table

,base models,stackers,meta,distinct 10-fold splits,fold models trained
preset,,,,,
low,4,1,0,5,50
medium,6,5,0,7,110
high,12,9,0,13,210
best,12,9,1,14,220


## 2. Replaying the splits for every dataset × resample

`run_benchmark2.py` builds each classifier with `random_state = resample id`. Constructing
`TSCGlueEnhancedV4(random_state=r, preset=p)` reproduces the exact `fold_seeds`; the stacker and
meta seeds are the next draws from the same generator in `_fit` (one draw per stacker sits between
them). For each split set we check whether any training fold lacks a class.

In [3]:
y_train = {
    (ds, r): load_from_ts_file(DATA / ds / f"{ds}{r}_TRAIN.ts")[1]
    for ds in DATASETS
    for r in RESAMPLES
}
print(f"{len(y_train)} training label vectors loaded")

1792 training label vectors loaded


In [4]:
def split_sets(clf, y):
    # Every 10-fold split one fit of `clf` would generate, in _fit order.
    X = np.zeros((len(y), 1))
    folds = lambda seed: generate_folds(X, y, n_splits=clf.k_folds, n_repetitions=1, random_state=seed)
    sets = {}
    for spec in clf.model_specs:
        for s in spec.fold_seeds:
            sets[("level0", spec.model_name)] = folds(s)
    sets[("level1", "stacker split")] = folds(clf._get_feature_seed())
    if clf.meta_stacking_models:
        for _ in clf.stacking_models:  # _fit_stacking_level draws one seed per stacker
            clf._get_feature_seed()
        sets[("level2", "meta split")] = folds(clf._get_feature_seed())
    return sets


def folds_missing_a_class(y, splits):
    n_classes = len(np.unique(y))
    return sum(len(np.unique(y[tr])) < n_classes for tr, _ in splits)


def replay(y, preset, r):
    if np.unique(y, return_counts=True)[1].min() < 2:
        return "singleton class"
    clf = TSCGlueEnhancedV4(random_state=r, preset=preset)
    with contextlib.redirect_stdout(io.StringIO()):  # silence get_folds' KFold message
        sets = split_sets(clf, y)
    for (level, _), splits in sets.items():
        if folds_missing_a_class(y, splits):
            return f"missing class in a {level} fold"
    return None


replayed = pd.DataFrame(
    [
        {"dataset": ds, "resample": r, "preset": p, "trigger": replay(y, p, r)}
        for (ds, r), y in y_train.items()
        for p in PRESETS
    ]
)
fallback_runs = replayed.dropna(subset=["trigger"])
n_runs = fallback_runs.groupby("dataset").size()
SYSTEMATIC = sorted(n_runs.index[n_runs == len(PRESETS) * len(RESAMPLES)])  # every run falls back
print("falls back on every run:", SYSTEMATIC)
print(fallback_runs[~fallback_runs["dataset"].isin(SYSTEMATIC)].to_string(index=False))
fallback_runs.pivot_table(index=["dataset", "trigger"], columns="preset", values="resample", aggfunc="count")

falls back on every run: ['DiatomSizeReduction', 'FiftyWords', 'Phoneme', 'PigAirwayPressure', 'PigArtPressure', 'PigCVP']
 dataset  resample preset                        trigger
FaceFour         2    low missing class in a level1 fold


,preset,best,high,low,medium
dataset,trigger,,,,
DiatomSizeReduction,singleton class,16.0,16.0,16.0,16.0
FaceFour,missing class in a level1 fold,NaN,NaN,1.0,NaN
FiftyWords,singleton class,16.0,16.0,16.0,16.0
Phoneme,singleton class,16.0,16.0,16.0,16.0
PigAirwayPressure,missing class in a level0 fold,16.0,16.0,16.0,16.0
PigArtPressure,missing class in a level0 fold,16.0,16.0,16.0,16.0
PigCVP,missing class in a level0 fold,16.0,16.0,16.0,16.0


Six datasets fall back on every resample and every preset, so for them the cause is structural,
not an unlucky seed. There is one extra run: FaceFour, `low` preset, one resample, at the level-1
stacker split. That is the latent `KFold` risk from section 3 actually happening.

**Check against the real runs.** A fallback serves `MRHydraET(random_state=r)` whatever the preset
and metric. A run fell back if its probabilities are *bit-identical* to those of a variant with a
different served head. For `low`/`medium`/`high` that means its Accuracy/LogLoss sibling. `best`
has no sibling here, so a `best`-only fallback would go undetected.

In [5]:
from tsml_eval.evaluation.storage import load_classifier_results


def probs(model, ds, r):
    return load_classifier_results(
        str(MODEL_RESULTS / model / "Predictions" / ds / f"testResample{r}.csv")
    ).probabilities


VARIANTS = {
    "low": ["Low-Accuracy", "Low-LogLoss"],
    "medium": ["Medium-Accuracy", "Medium-LogLoss"],
    "high": ["High-Accuracy", "High-LogLoss"],
    "best": ["Best-LogLoss"],
}
observed = set()
for ds in DATASETS:
    for r in RESAMPLES:
        P = {v: probs(f"TSCGlueEnhancedV4-{v}-GPU", ds, r) for vs in VARIANTS.values() for v in vs}
        for preset, vs in VARIANTS.items():
            if any(np.array_equal(P[v], P[w]) for v in vs for w in P if w != v):
                observed.add((ds, r, preset))

predicted = set(fallback_runs[["dataset", "resample", "preset"]].itertuples(index=False, name=None))
print(f"replay predicts fallback  : {len(predicted)} runs")
print(f"fallback in model_results : {len(observed)} runs")
print(f"sets equal                : {predicted == observed}")

replay predicts fallback  : 385 runs
fallback in model_results : 385 runs
sets equal                : True


## 3. Why these datasets

There are two independent root causes.

In [6]:
counts = pd.DataFrame(
    [
        {
            "dataset": ds,
            "n_train": len(y),
            "n_classes": len(c),
            "min per class": c.min(),
            "max per class": c.max(),
            "singleton classes": int((c == 1).sum()),
        }
        for (ds, r), y in y_train.items()
        if r == 0
        for c in [np.unique(y, return_counts=True)[1]]
    ]
).set_index("dataset")
counts["split used"] = np.where(counts["max per class"] >= K, "StratifiedKFold", "KFold")
counts["fallback runs (of 64)"] = n_runs.reindex(counts.index, fill_value=0)
counts[(counts["min per class"] < 2) | (counts["split used"] == "KFold")].sort_values("fallback runs (of 64)")

,n_train,n_classes,min per class,max per class,singleton classes,split used,fallback runs (of 64)
dataset,,,,,,,
Beef,30,5,6,6,0,KFold,0
InsectEPGSmallTrain,17,3,3,8,0,KFold,0
Symbols,25,6,3,8,0,KFold,0
Rock,20,4,5,5,0,KFold,0
FaceFour,24,4,3,8,0,KFold,1
Phoneme,214,39,1,24,8,StratifiedKFold,64
FiftyWords,450,50,1,52,1,StratifiedKFold,64
DiatomSizeReduction,16,4,1,6,1,KFold,64
PigArtPressure,104,52,2,2,0,KFold,64


**Cause A: singleton classes (DiatomSizeReduction, FiftyWords, Phoneme).** A class with one
training case is in the validation part of some fold, so that fold's training part cannot contain
it. `_fit` does not even try: the `counts < 2` guard sends these straight to the fallback.

**Cause B: `get_folds` silently degrades to plain `KFold` (PigAirwayPressure, PigArtPressure, PigCVP).**
sklearn's `StratifiedKFold` raises `ValueError` when `n_splits` exceeds the size of *every*
class. It only warns when *some* classes are smaller than `n_splits`. The Pig datasets have
52 classes × 2 cases, so it raises and `get_folds` switches to unstratified `KFold`. With 104
cases spread over 10 validation folds of ~10, some class almost surely has both cases in the same
validation fold. That fold's model never sees the class → fallback. Stratification would have
avoided this: a class with 2 cases would go to 2 different folds, so every training part keeps one.

The other five `KFold` datasets (Beef, FaceFour, InsectEPGSmallTrain, Rock, Symbols) have at least
3 cases per class, so a class disappears from a training fold only when all its cases land in the
same validation fold of 2–3 cases. That is rare, but not impossible: FaceFour hit it once.

In [7]:
y_pig = y_train[("PigCVP", 0)]
try:
    StratifiedKFold(n_splits=K, shuffle=True, random_state=0).split(np.zeros(len(y_pig)), y_pig).__next__()
except ValueError as e:
    print("StratifiedKFold:", e)

with contextlib.redirect_stdout(io.StringIO()):
    kf = tg_utils.get_folds(np.zeros(len(y_pig)), y_pig, n_splits=K, random_state=0)
print("PigCVP, one KFold split: classes missing from each training fold ->",
      [len(set(y_pig)) - len(set(y_pig[tr])) for tr, _ in kf])

StratifiedKFold: n_splits=10 cannot be greater than the number of members in each class.
PigCVP, one KFold split: classes missing from each training fold -> [0, 0, 1, 0, 2, 2, 0, 0, 0, 0]


In [8]:
# Latent risk on the surviving KFold datasets: P(one split set misses a class), over random seeds,
# and P(a whole fit falls back) given the number of split sets per preset (table in section 1).
N_SEEDS = 2000
n_sets = preset_table["distinct 10-fold splits"].to_dict()
risk = []
for ds in counts.index[(counts["split used"] == "KFold") & (counts["min per class"] >= 3)]:
    y = y_train[(ds, 0)]
    with contextlib.redirect_stdout(io.StringIO()):
        p = np.mean([
            folds_missing_a_class(y, tg_utils.get_folds(np.zeros(len(y)), y, n_splits=K, random_state=s)) > 0
            for s in range(N_SEEDS)
        ])
    risk.append({"dataset": ds, "P(split misses a class)": p}
                | {f"P(fit falls back, {k})": 1 - (1 - p) ** n for k, n in n_sets.items()})
pd.DataFrame(risk).set_index("dataset").round(4)

,P(split misses a class),"P(fit falls back, low)","P(fit falls back, medium)","P(fit falls back, high)","P(fit falls back, best)"
dataset,,,,,
Beef,0.0000,0.0000,0.0000,0.0000,0.0000
FaceFour,0.0015,0.0075,0.0105,0.0193,0.0208
InsectEPGSmallTrain,0.0000,0.0000,0.0000,0.0000,0.0000
Rock,0.0000,0.0000,0.0000,0.0000,0.0000
Symbols,0.0075,0.0369,0.0513,0.0932,0.1000


## 4. What the fallback costs

Mean accuracy over resamples 0–5 (the resamples every classifier has), and mean rank among all
benchmarked classifiers: on the six systematic fallback datasets vs. on the other 106.

In [9]:
FALLBACK_DS = SYSTEMATIC
models = sorted(p.name for p in MODEL_RESULTS.iterdir() if p.is_dir() and p.name != "logs")
acc = pd.DataFrame(
    {
        m: {
            ds: np.mean([load_classifier_results(
                str(MODEL_RESULTS / m / "Predictions" / ds / f"testResample{r}.csv")).accuracy
                for r in range(6)])
            for ds in DATASETS
        }
        for m in models
    }
)
acc.loc[FALLBACK_DS].T.round(3).sort_values("PigCVP", ascending=False)

,DiatomSizeReduction,FiftyWords,Phoneme,PigAirwayPressure,PigArtPressure,PigCVP
WEASEL_V2,0.943,0.814,0.334,0.887,0.962,0.950
HIVECOTEV2,0.932,0.820,0.398,0.961,0.982,0.941
RDST,0.935,0.832,0.332,0.796,0.944,0.905
ROCKET,0.958,0.821,0.274,0.389,0.911,0.885
MultiRocketHydra,0.949,0.843,0.360,0.711,0.939,0.868
TSCGlueEnhancedV4-Best-LogLoss-GPU,0.934,0.751,0.369,0.773,0.949,0.838
TSCGlueEnhancedV4-High-LogLoss-GPU,0.934,0.751,0.369,0.773,0.949,0.838
TSCGlueEnhancedV4-Low-Accuracy-GPU,0.934,0.751,0.369,0.773,0.949,0.838
TSCGlueEnhancedV4-Low-LogLoss-GPU,0.934,0.751,0.369,0.773,0.949,0.838
TSCGlueEnhancedV4-Medium-Accuracy-GPU,0.934,0.751,0.369,0.773,0.949,0.838


In [10]:
ranks = acc.rank(axis=1, ascending=False)
pd.DataFrame(
    {
        "mean rank, 6 fallback datasets": ranks.loc[FALLBACK_DS].mean(),
        "mean rank, other 106 datasets": ranks.drop(index=FALLBACK_DS).mean(),
    }
).round(2).sort_values("mean rank, other 106 datasets")

,"mean rank, 6 fallback datasets","mean rank, other 106 datasets"
TSCGlueEnhancedV4-High-Accuracy-GPU,7.50,4.70
TSCGlueEnhancedV4-Medium-Accuracy-GPU,7.50,4.82
TSCGlueEnhancedV4-Best-LogLoss-GPU,7.50,5.15
TSCGlueEnhancedV4-Medium-LogLoss-GPU,7.50,5.58
TSCGlueEnhancedV4-High-LogLoss-GPU,7.50,5.63
TSCGlueEnhancedV4-Low-Accuracy-GPU,7.50,6.25
HIVECOTEV2,3.50,6.96
TSCGlueEnhancedV4-Low-LogLoss-GPU,7.50,7.55
MultiRocketHydra,6.67,8.04
RDST,5.83,9.68


Every TSCGlue preset is the best or near-best classifier on the normal datasets, but on these six
all seven variants tie at the fallback's rank, behind HC2, WEASEL_V2, RDST and MultiRocketHydra.
WEASEL and RDST are TSCGlue base models, so the stack has the representations that win on these
datasets (e.g. PigCVP: WEASEL_V2 0.95, RDST 0.91 vs. fallback 0.84); it just never gets to use them.
The Pig runs also waste time: all features are extracted before the first fold model reports the
missing class.

## 5. Can it be avoided?

Yes. Neither cause is a real limitation of the stack.

**Fix B: stratify for any `k`.** sklearn raises only as a guard; round-robin assignment of each class
over the folds works for any class size. A class with *c ≥ 2* cases lands in min(c, k) different
validation folds, so every training part keeps at least one case. Replacing `get_folds` with:

In [11]:
def stratified_folds(y, n_splits, random_state):
    # Shuffled round-robin per class; the running offset keeps fold sizes balanced.
    rng = np.random.default_rng(random_state)
    fold_of = np.empty(len(y), dtype=int)
    offset = 0
    for cls in rng.permutation(np.unique(y)):
        idx = rng.permutation(np.flatnonzero(y == cls))
        fold_of[idx] = (offset + np.arange(len(idx))) % n_splits
        offset += len(idx)
    return [(np.flatnonzero(fold_of != k), np.flatnonzero(fold_of == k)) for k in range(n_splits)]


fixed = []
for (ds, r), y in y_train.items():
    splits = stratified_folds(y, K, r)
    sizes = [len(va) for _, va in splits]
    fixed.append({
        "dataset": ds, "resample": r,
        "folds missing a class": folds_missing_a_class(y, splits),
        "val fold size spread": max(sizes) - min(sizes),
    })
fixed = pd.DataFrame(fixed)
print("max validation-fold size spread:", fixed["val fold size spread"].max())
fixed[fixed["folds missing a class"] > 0].groupby("dataset")["folds missing a class"].agg(["count", "max"])

max validation-fold size spread: 1


,count,max
dataset,,
DiatomSizeReduction,16,1
FiftyWords,16,1
Phoneme,16,7


With stratified folds the Pig datasets and the five at-risk `KFold` datasets never miss a class.
Only the three singleton datasets are left, each with exactly one fold per singleton class.

**Fix A: singleton classes.** No split can put a single case in both the training and the
validation part. Two ways to keep the stack:

- *A1: pad.* Align the fold model's `predict_proba` to `self.classes_`, with 0 for the class it did
  not see. Mechanically this is enough: the level-1/2 stackers (Ridge, LogisticCV with its singleton
  guard, ET, RF, MLP without early stopping) all fit with a singleton class present. The weakness
  is that the singleton's only OOF row gives 0 probability to its own class. The stacker has no
  example of "high probability for S → S", so it will rarely predict S, even though 9 of the 10
  base fold models learned it.
- *A2: duplicate singleton rows before splitting.* Each singleton then has 2 cases, stratified
  folds put them in different folds, every fold model sees the class, and the stacker gets 2 OOF
  rows with a meaningful probability for S. The cost is optimistic OOF for those 2 rows (each is
  predicted by a model trained on its twin), which only affects the singleton class.

A1 loses at most the test cases of singleton classes. A2 should lose far fewer, but that needs a
refit to confirm. Share of the test set in singleton classes:

In [12]:
share = []
for ds in counts.index[counts["singleton classes"] > 0]:
    for r in RESAMPLES:
        cls, c = np.unique(y_train[(ds, r)], return_counts=True)
        y_test = load_from_ts_file(DATA / ds / f"{ds}{r}_TEST.ts")[1]
        share.append({"dataset": ds, "test share in singleton classes": np.isin(y_test, cls[c == 1]).mean()})
pd.DataFrame(share).groupby("dataset").agg(["mean", "max"]).round(4)

test share in singleton classes        
                                               mean     max
dataset                                                    
DiatomSizeReduction                          0.1078  0.1078
FiftyWords                                   0.0132  0.0132
Phoneme                                      0.0269  0.0269

On FiftyWords (fallback 0.751 vs. MultiRocketHydra 0.843) even the A1 worst case is a clear win.
On DiatomSizeReduction (fallback 0.934, best 0.958) A1's worst case of 10.8% outweighs the upside,
so A2 is the variant to try first.

**Fix C (cheap, independent): decide before extracting features.** The splits depend only on `y`
and the seeds, so all of them can be generated and checked at the start of `_fit`. Wherever a
fallback is still needed, it then costs one MRHydraET fit instead of a full feature extraction
plus a fallback fit.

## Conclusions

- **10-fold mechanics.** Transformers are fit once, unsupervised, on the full training set. Each
  base model is 10-fold trained on its own split, then each stacker on a new split over the OOF
  probabilities (plus a meta split for `best`), so a fit has 5–14 independent splits and 50–220
  fold models. Any fold model that misses a class, or any class with a single case, abandons the
  whole stack for `MRHydraET`.
- **Where the fallback runs.** On 6 of 112 UCR datasets, on every resample and every preset, plus
  one FaceFour `low` run. The offline replay matches the fallbacks visible in `model_results`
  exactly.
  - DiatomSizeReduction, FiftyWords, Phoneme: singleton classes.
  - PigAirwayPressure, PigArtPressure, PigCVP (and the FaceFour run): `get_folds` catches
    `StratifiedKFold`'s `ValueError` and silently uses unstratified `KFold`.
- **Cost.** On these datasets TSCGlue drops from the top ranks to mid-pack, behind classifiers
  whose representations it already contains. All 7 TSCGlue variants tie there.
- **Improvement.**
  - Replace `get_folds` with class-wise round-robin stratification. This removes the Pig and
    FaceFour fallbacks and the latent risk on the other small `KFold` datasets, and can be verified
    without refitting.
  - For singleton classes, duplicate the singleton rows (A2) rather than only padding (A1).
  - Check splits before feature extraction.
  - Next step: refit the six datasets with the patch to measure the actual gain.